Ridership ≈ Service × Demand × Accessibility

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
pd.set_option('display.max_columns', None)
import os
import google.auth
import gcsfs
fs = gcsfs.GCSFileSystem()
import statsmodels.api as sm
import numpy as np
from scipy.stats import skew


import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
GCS_FILE_PATH = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
with fs.open(f"{GCS_FILE_PATH}/stop_route_df.parquet", "rb") as f: 
    stop_route_df = gpd.read_parquet(f)

In [5]:
stop_route_df.head(2)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry,total_pop_adj,poverty_pop_adj,non_us_citizen_adj,workers_with_no_car_adj,households_with_no_cars_adj,disabled_pop_adj,public_asst_pop_adj,inc_extremelylow_adj,inc_verylow_adj,inc_low_adj,male_seniors_adj,female_seniors_adj,veteran_pop_adj,male_youth_adj,inc_total_lowincome_adj,female_youth_adj,total_seniors_adj,jobs_tot_adj,total_youth_adj,ALAND_adj
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,VNACLR1,.,None,14.0,1.0,POINT(-119.294028 34.343645),Weekday,0.00000,3.000000,2025-05-01,2025-05-31,POINT (64936.582 -407800.219),41.453354,3.949242,3.158169,0.424528,0.463815,5.709516,2.163260,9.215728,9.896572,3.552156,3.815590,4.852721,1.841532,1.880153,22.664456,2.773107,8.668311,26.318716,4.653260,1.969641e+06
1,Samtrans,db97cc02836aa5f0cf647d80160b23ec,345017,1000 El Camino Real-Menlo College,345017,72.0,1.0,POINT(-122.191284 37.457543),Weekday,9.52381,24.571429,2025-08-01,2025-08-31,POINT (-193547.746 -59885.338),2537.657815,137.769913,429.179726,163.623904,133.342872,193.085414,49.337153,357.458768,290.410532,176.891016,198.211651,295.773462,68.365482,145.896194,824.760316,253.752229,493.985113,1266.679117,399.648424,2.028257e+06


In [6]:
# Convert n_arrivals and n_routes to integer
stop_route_df['n_arrivals'] = stop_route_df['n_arrivals'].fillna(0).astype(int)
stop_route_df['n_routes'] = stop_route_df['n_routes'].fillna(0).astype(int)

In [7]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 21264 entries, 0 to 21263
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   organization_name            21264 non-null  object  
 1   feed_key                     21264 non-null  object  
 2   stop_id                      21264 non-null  object  
 3   stop_name                    21264 non-null  object  
 4   stop_code                    20543 non-null  object  
 5   n_arrivals                   21264 non-null  int64   
 6   n_routes                     21264 non-null  int64   
 7   pt_geom                      21264 non-null  object  
 8   day_type                     21264 non-null  object  
 9   average_daily_boardings      21264 non-null  float64 
 10  average_daily_alightings     19087 non-null  float64 
 11  start_date                   21264 non-null  object  
 12  end_date                     21264 non-null  object 

### Log-linear Regression Model

In [8]:
# Log-linear regression
# Dependent variable: log('average_daily_boardings')
# Predictors: n_trips, n_routes, total_pop_adj, households_with_no_cars_adj, jobs_tot_adj

# Copy the dataframe
df = stop_route_df.copy()

# 2. Create log dependent variable
# Replace 0 with NaN BEFORE log (log(0) = -inf)
df['log_boardings'] = np.log(df['average_daily_boardings'].replace(0, np.nan))

# 3. Select only rows with all needed variables
df = df.dropna(subset=[
    'average_daily_boardings',
    'n_arrivals',
    'n_routes',
    'total_pop_adj',
    'workers_with_no_car_adj'
])

# 4. Define Y and X
y = df['average_daily_boardings']

X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'workers_with_no_car_adj']]

# 5. Add constant
X = sm.add_constant(X)

# 6. Fit model
model = sm.OLS(y, X).fit()

# 7. Show results
print(model.summary())

                               OLS Regression Results                              
Dep. Variable:     average_daily_boardings   R-squared:                       0.056
Model:                                 OLS   Adj. R-squared:                  0.056
Method:                      Least Squares   F-statistic:                     313.7
Date:                     Thu, 09 Apr 2026   Prob (F-statistic):          1.16e-262
Time:                             19:03:47   Log-Likelihood:            -1.5966e+05
No. Observations:                    21264   AIC:                         3.193e+05
Df Residuals:                        21259   BIC:                         3.194e+05
Df Model:                                4                                         
Covariance Type:                 nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------

- About 5.6% of the variation in dependent variable (here, stop-level ridership) is explained by the predictors in the model.
- 94.4% of the variation is due to other factors not captured by this model.
- Skew = 54.357 → Residuals are extremely asymmetric, violating normality.
- Kurtosis = 4235.968 → Residuals have ultra‑heavy tails, far from a normal distribution.
- JB statistic = 15 billion → The Jarque–Bera test overwhelmingly rejects normality.
- Durbin–Watson = 1.374 → Strong positive autocorrelation remains in the residuals.
- Condition number = 1.92e+04 → Predictors have scaling issues and possible multicollinearity.

### Checking Multicollinearity using VIF

VIF checks multicollinearity (correlation among predictors)

- Idea:
If a variable can be well explained by other predictors,
its coefficient becomes unstable and its standard error increases
- Interpretation:
VIF ≈ 1   → no correlation (good);
VIF > 5   → moderate concern;
VIF > 10  → serious multicollinearity
- Key point:
High VIF doesn't bias coefficients, but makes them noisy and hard to interpret

In [9]:
# Selecting only numeric regressors (Ridership ≈ Service × Demand × Accessibility)
X = stop_route_df[
    ['n_arrivals', 'n_routes',  'total_pop_adj', 'workers_with_no_car_adj']
].copy()

# Add constant
X = sm.add_constant(X)

vif_df = pd.DataFrame()
vif_df["variable"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif_df)

                  variable       VIF
0                    const  6.623978
1               n_arrivals  1.527425
2                 n_routes  1.420235
3            total_pop_adj  1.701594
4  workers_with_no_car_adj  1.610496


All predictors have low VIFs (≈1–1.8), which means very little multicollinearity among chosen variables.
The model’s coefficients should therefore be stable and interpretable, with no inflation of standard errors.
The high VIF on the constant is not meaningful and can be ignored.

In [10]:
stop_route_df['average_daily_boardings'].describe()

count    21264.000000
mean        43.375367
std        454.066948
min          0.000000
25%          2.000000
50%          6.880785
75%         21.795550
max      41893.868537
Name: average_daily_boardings, dtype: float64

Data is highly skewed (28.63) .
Skewed data is data that isn’t evenly distributed around the center, values bunch up on one side and stretch into a long tail.
This means a few stops have very high boardings and push the distribution out into a long tail.
75% of stops board ≤ 20 people

Possible solution: Use a count model (Poisson or Negative Binomial)  
Other models to check:
- Poisson
- Negative Binomial (handles overdispersion better)
- Zero‑inflated Poisson (if many zeros)
- Zero‑inflated NB (most flexible)

### Poisson Model

In [11]:
# Copy the dataset
df = stop_route_df.copy()

# 2. Select needed variables (drop missing)
df = df.dropna(subset=[
    'average_daily_boardings',
    'n_arrivals', 'n_routes',
    'total_pop_adj', 'workers_with_no_car_adj', 'poverty_pop_adj'
])

# 3. Define Y and X
y = df['average_daily_boardings']

X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'workers_with_no_car_adj', 'poverty_pop_adj']]

# 4. Add intercept
X = sm.add_constant(X)

# 5. Fit Poisson GLM
poisson_model = sm.GLM(y, X, family=sm.families.Poisson()).fit()

# 6. Output summary
print(poisson_model.summary())


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:                21264
Model:                                 GLM   Df Residuals:                    21258
Model Family:                      Poisson   Df Model:                            5
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:            -1.3658e+06
Date:                     Thu, 09 Apr 2026   Deviance:                   2.6532e+06
Time:                             19:05:18   Pearson chi2:                 3.09e+07
No. Iterations:                         14   Pseudo R-squ. (CS):              1.000
Covariance Type:                 nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

This Poisson model fits very poorly: the Pearson chi‑square is huge, meaning the data are heavily over‑dispersed and the Poisson assumptions break.
- The coefficient for n_arrivals (0.0253) means each extra trip increases expected ridership by about 2.5%.
- n_routes (-1.3051) is negative, which is unrealistic and indicates model misspecification due to over‑dispersion.
- total_pop_adj is negative, also unrealistic — another sign the Poisson model is not appropriate.
- workers_with_no_car_adj is positive, meaning more zero‑car households predict higher ridership.
- poverty_pop_adj is positive, meaning as the adjusted population living below the poverty level increases in a transit agency’s service area, ridership tends to increase.
Overall: the Poisson model’s signs are distorted.

Pearson chi‑square = 3.09e+07 and Degrees of freedom ≈ 21264.
Hence, 
Dispersion estimate = Pearson χ² / df; 3.09e+07 / 21264 ≈ 1453.5

A valid Poisson model should have dispersion ≈ 1. Dispersion ≈ 1453.5, which means:
Poisson model is severely overdispersed. >1 → overdispersion (variance is larger than the model assumes).

Variance >> mean:
- Poisson standard errors become too small
- p‑values become artificially tiny
- coefficients look falsely “precise”
- model fit is misleading"

### Negative Binomial Model

In [12]:
X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'workers_with_no_car_adj',  'total_youth_adj',  'total_seniors_adj']]

# Same X and y as  Poisson
nb_model_extended = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial()
).fit()

print(nb_model_extended.summary())

/opt/conda/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:                21264
Model:                                 GLM   Df Residuals:                    21258
Model Family:             NegativeBinomial   Df Model:                            5
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:                -83586.
Date:                     Thu, 09 Apr 2026   Deviance:                       39854.
Time:                             19:06:55   Pearson chi2:                 1.63e+05
No. Iterations:                         30   Pseudo R-squ. (CS):             0.8175
Covariance Type:                 nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

### Negative Binomial Model 

The Negative Binomial model behaves much better than the Poisson model because it corrects the strongly overdispersed nature of ridership data. After switching to a Negative Binomial model, the dispersion dropped to a reasonable level (≈5), the standard errors became realistic, the z‑values returned to interpretable ranges, and the overall fit stabilized. 


The NB model is used when:
- counts vary wildly
- some stops have way more riders than others
- data is highly skewed
- there are outliers or “hot spots”

Variance > Mean
This matches real transit ridership perfectly.

In the above model,
Pseudo R² (Cragg‑Uhler / Nagelkerke): 0.8978
Very high — the predictors explain a large proportion of the variation in ridership.
For transit ridership data (often noisy and over‑dispersed), this is exceptionally strong.



### Variables Interpretation

1. n_arrivals coefficient = 0.0425
Interpretation: Each additional trip serving the stop increases expected daily boardings by:
exp(0.0425) − 1 ≈ 4.34%

2. n_routes coefficient = 0.1426
Interpretation: For stops/agencies where n_routes_effective > 0 (aggregated stop-level data), each additional route increases expected daily boardings by:
exp(0.1426)−1 ≈ 15.34%



3. total_pop_adj – Population in stop buffer
total_pop_adj represents the estimated number of residents physically inside a stop’s 600 m buffer, after adjusting each census tract by the fraction of its area that overlaps the buffer.
Example: if 10% of a tract lies inside a stop’s buffer, 10% of the tract’s population is assigned to that stop.
Coefficient interpretation:
Model coefficient: 8.518e-05
Effect per additional person: exp(8.518e-05)−1≈0.0085% increase in expected daily boardings per person

  This looks tiny for one person, but stop buffers usually contain hundreds or thousands of people.
     Scaling to realistic population changes:

    +1,000 residents in the stop buffer: exp(1000×8.518e-05)−1 ≈ 8.89% increase in ridership
    +10,000 residents in the stop buffer: exp(10,000×8.518e-05)−1 ≈ 134.4% increase in ridership

4. workers_with_no_car_adj (coef = 0.0002)
workers_with_no_car_adj is the estimated number of population that do not own a car within the stop catchment, computed the same area-weighted way.

For each additional zero-car population in the catchment:
exp(0.0002) – 1 ≈ 0.02% increase in expected boardings

- +100 zero-car population in buffer : ≈ 2.02% ridership increase
- +1,000 zero-car population : ≈ 22.1% ridership increase

5. total_youth_adj (coef = 0.0003)
total_youth_adj is the estimated number of youth population within the stop catchment, computed the same area-weighted way.

For each additional youth population in the catchment:
exp(0.0003) – 1 ≈ 0.03% increase in expected boardings

- +100 youths in buffer : ≈ 3.05% ridership increase
- +1,000 youths in buffer : ≈ 35.0% ridership increase

5. total_senior_adj (coef = 6.758e-06)
total_senior_adj is the estimated number of senior population within the stop catchment, computed the same area-weighted way.

For each additional senior population in the catchment:
exp(6.758e-06) – 1 ≈ 0.000676% change in expected boardings

- +100 seniors in buffer : ≈ 0.0676% ridership increase
- +1,000 seniors in buffer : ≈ 0.678% ridership increase
(Negligible effect),

p = 0.782 → far greater than 0.05, meaning not statistically significant.

## Negative Binomial GLM (Log‑Link) Model Equation

The model estimates the expected daily boardings at stop *i* using a Negative Binomial
Generalized Linear Model with a log link.

The linear predictor is:

$$
\log(\mu_i) =
\beta_0
+ \beta_1 \, (\text{n\_trips}_i)
+ \beta_2 \, (\text{n\_routes}_i)
+ \beta_3 \, (\text{total\_pop\_adj}_i)
+ \beta_4 \, (\text{workers\_with\_no\_cars\_adj}_i)
+ \beta_5 \, (\text{total\_youth\_adj}_i)
+ \beta_7 \, (\text{total\_seniors\_adj}_i)
$$

Where the expected value of daily boardings is:

$$
\mu_i = E[\text{average\_daily\_boardings}_i]
$$

The outcome follows a Negative Binomial distribution:

$$
Y_i \sim \text{NB}(\mu_i, \alpha)
$$

The log link implies that predictor effects are multiplicative:

$$
\mu_i = \exp\left(
\beta_0
+ \beta_1 \, \text{n\_trips}_i
+ \beta_2 \, \text{n\_routes}_i
+ \beta_3 \, \text{total\_pop\_adj}_i
+ \beta_4 \, \text{workers\_with\_no\_cars\_adj}_i
+ \beta_5 \, \text{total\_youth\_adj}_i
+ \beta_7 \, \text{total\_seniors\_adj}_i
\right)
$$

Statsmodels’ `GLM(..., family=NegativeBinomial())` uses a fixed dispersion parameter $\alpha$
unless otherwise specified.

In log_linear ols regression model Y is continuous variable i.e. if n_trip has coefficient of 0.1 it means A one‑unit increase in trips increases the geometric mean of ridership by 10%.
While in this model: it means Each additional daily trip increases expected ridership by about 10%.

1) Log-linear OLS model
Modeling log(ridership). 
When the coefficient is 0.1, it means:
→ adding 1 trip increases ridership by about 10% on average (geometric mean).
2) Negative Binomial model
This model is built for count data (like number of riders).
A coefficient of 0.1 also means about a 10% increase, but interpreted as:
→ each extra trip increases the expected number of riders by 10%.

They sound almost identical, but:
- Log-linear OLS focuses on modeling a transformed version of ridership (log scale). Talks about the geometric mean (a bit abstract)
Negative Binomial
- directly models counts (actual riders). Talks about expected ridership (more natural and realistic for counts)

### Other Tests 

Total Jobs variable added

In [13]:
# Selecting only numeric regressors (Ridership ≈ Service × Demand × Accessibility)
X = stop_route_df[
    ['n_arrivals', 'n_routes',  'total_pop_adj', 'workers_with_no_car_adj', 'total_youth_adj',  'total_seniors_adj', 'jobs_tot_adj']
].copy()

# Add constant
X = sm.add_constant(X)

vif_df = pd.DataFrame()
vif_df["variable"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif_df)

                  variable        VIF
0                    const   6.991227
1               n_arrivals   1.540762
2                 n_routes   1.427478
3            total_pop_adj  28.958436
4  workers_with_no_car_adj   1.632586
5          total_youth_adj   2.653703
6        total_seniors_adj   3.405191
7             jobs_tot_adj  22.094876


In [14]:
X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'workers_with_no_car_adj',  'total_youth_adj',  'total_seniors_adj', 'jobs_tot_adj']]

# Same X and y as  Poisson
nb_model_extended = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial()
).fit()

print(nb_model_extended.summary())

/opt/conda/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:                21264
Model:                                 GLM   Df Residuals:                    21257
Model Family:             NegativeBinomial   Df Model:                            6
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:                -83560.
Date:                     Thu, 09 Apr 2026   Deviance:                       39802.
Time:                             19:16:01   Pearson chi2:                 1.64e+05
No. Iterations:                         30   Pseudo R-squ. (CS):             0.8179
Covariance Type:                 nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

The addition of `jobs_tot_adj` into the model introduces **substantial multicollinearity**, as evidenced by very high VIF values for both `total_pop_adj` (VIF ≈ 28.96) and `jobs_tot_adj` (VIF ≈ 22.09). This indicates that these two variables are highly correlated, reflecting overlapping spatial characteristics: areas with higher population tend to also have higher employment.

As a result, the model struggles to disentangle their individual effects. Specifically:

`total_pop_adj` now has a coefficient of 8.343e-06 with a p-value of 0.416, making it statistically insignificant. Its effect on ridership cannot be reliably interpreted in the presence of `jobs_tot_adj`'.
`jobs_tot_adj` enters as a positive and significant predictor (coef = 0.0002, p < 0.001), absorbing much of the explanatory power that `total_pop_adj` previously provided.

Other predictors, including 'n_arrivals', 'n_routes', and 'total_youth_adj', remain significant with interpretable effects, while 'total_seniors_adj' remains statistically insignificant.

Despite only a marginal improvement in overall model fit (Pseudo R² increasing slightly from 0.8175 to 0.8179), the interpretability of individual coefficients, particularly for `total_pop_adj`, is reduced. In this specification, employment and population do not independently contribute to ridership; rather, they compete to explain the same underlying spatial variation. This demonstrates the importance of considering multicollinearity when interpreting coefficients in spatial ridership models.